In [ ]:
# test from (1 ... 18)
n_images = {n_images_value}

In [ ]:
from astroquery.mast.missions import MastMissions, conf
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u
from glob import glob
import os

In [ ]:
def use_mast_test(mast):
    mast._service_api_connection.SERVICE_URL = 'https://masttest.stsci.edu'
    mast._service_api_connection.REQUEST_URL = 'https://masttest.stsci.edu/search/roman/api/v0.1/'
    mast._service_api_connection.MISSIONS_DOWNLOAD_URL = 'https://masttest.stsci.edu/search/'

mast_token = os.environ.get('MAST_TEST_TOKEN')
conf.server = "https://masttest.stsci.edu"
mm = MastMissions(mission='roman')
mm.login(mast_token)
use_mast_test(mm)

print(f"✓ Authenticated with masttest (token: {mast_token[:8]}...)")

In [ ]:
# Check for locally cached files for L2s from 18 Roman detectors. If none are found, download them.
existing_downloads = glob(
    'mastDownload/roman/r0000101001001001001_0002*/*'
)

if len(existing_downloads) != 18:
    coord = SkyCoord(ra=270.019512 * u.deg, dec=65.9667761 * u.deg)
    multiple_observations = mm.query_criteria(
        coordinates=coord, 
        radius=30*u.arcmin, 
        optical_element='F158', 
        product_type='l2', 
        exposure_type='WFI_IMAGE'
    )
    mask = ['r0000101001001001001_0002_wfi' in obs['fileSetName'] for obs in multiple_observations]
    one_exposure = multiple_observations[mask]
    assert len(one_exposure) == 18
    product_list = mm.get_product_list(one_exposure)
    cal_files = mm.filter_products(product_list, file_suffix='_cal')
    downloads = mm.download_products(cal_files)
else: 
    downloads = {'Local Path': [path for path in existing_downloads]}

print(f"Found {len(downloads['Local Path'])} Roman SCA files")

In [ ]:
# Initialize MAST Aladin:
from mast_aladin import MastAladin

ma = MastAladin(
    target="270.019512 65.9667761",
    fov=60,
    sidecar='split-right'
)
ma

In [ ]:
# Batch load n_images Roman ASDF files into MAST Aladin
for i, path in enumerate(sorted(downloads['Local Path'])[:n_images], 1):
    ma.add_asdf(
        path,
        name=f"SCA_{i:02d}",
        opacity=0.7
    )